In [0]:
from pyspark.sql import functions as F
from config.f1_config import CATALOG, SILVER_SCHEMA, GOLD_SCHEMA
GOLD_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.races"

In [0]:
df_silver_races = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.races")
df_silver_circuits = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.circuits")


In [0]:
def build_dim_races(df_silver_races, df_silver_circuits):
    return(
        df_silver_races 
        .join(df_silver_circuits, on="Circuit_Id", how ="left")
        .select(
            df_silver_races["Season"],
            df_silver_races["Round"],
            df_silver_races["Race_Name"],
            df_silver_races["Race_Date"],
            df_silver_races["Race_Time"],
            df_silver_circuits["Circuit_Id"],
            df_silver_circuits["Circuit_Name"], 
            df_silver_circuits["Country"] )
      
)
df_gold_dim_races = build_dim_races(df_silver_races, df_silver_circuits)


In [0]:
df_gold_dim_races.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(GOLD_TABLE)